In [0]:
%pip install websocket-client confluent-kafka

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import col

In [0]:
# Cấu hình Confluent Kafka
KAFKA_SERVER = "YOUR_BOOTSTRAP_SERVER"
KAFKA_API_KEY = "YOUR_API_KEY"
KAFKA_API_SECRET = "YOUR_API_SECRET"
TOPICS = "binance-kline-1m,binance-trades"

## Get data from Kafka

In [0]:
sasl_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";'

# Kline data 
df_kafka = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVER)
    .option("subscribe", TOPICS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", sasl_config)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 50000)
    .load())

df_kline = df_kafka \
    .filter(col("topic") == "binance-kline-1m") \
    .selectExpr("CAST(value AS STRING) as json_payload")

kline_path = "/Volumes/workspace/bronze/binance_raw/kline_1m"
kline_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/01_ingest/kline_1m"

query_kline = (df_kline.writeStream
    .format("text") 
    .option("checkpointLocation", kline_checkpoint)
    .option("path", kline_path)
    .trigger(availableNow=True)
    .start()
)

# Trades data
df_trades = df_kafka \
    .filter(col("topic") == "binance-trades") \
    .selectExpr("CAST(value AS STRING) as json_payload")

trades_path = "/Volumes/workspace/bronze/binance_raw/trades"
trades_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/01_ingest/trades"

query_trades = (df_trades.writeStream
    .format("text") 
    .option("checkpointLocation", trades_checkpoint)
    .option("path", trades_path)
    .trigger(availableNow=True)
    .start()
)

query_kline.awaitTermination()
query_trades.awaitTermination()
print("Đã xả file thô thành công vào 2 thư mục riêng biệt trong Volume!")